In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import synxm 
from pathlib import Path

parameters = {
    "channels": ["PRRT", "A1", "PSD", "RIM"],
    "detection": {
        "PRRT": {"scale_um": 0.4, "specificity": 6},
        "A1": {"scale_um": 0.3, "specificity": 6},
        "PSD": {"scale_um": 0.1, "specificity": 8},
        "RIM": {"scale_um": 0.1, "specificity": 6},
    },
    "dbscan": {"channels": ["RIM", "PSD"],"eps": 1.0, "min_samples": 3},
    "synapse": {"distance": 1.0, "channels": ["RIM", "PSD"]},
    "pairs": {"distance": 0.5, "channels": ["A1", "PRRT"]},
}

output_folder = Path("./results")

folder = [
    "/media/pandora/Greger_group/Xinyao/PRRT Revision/PRRTm750 A1rb647 PSDck555 RIMgp488/261006 Re-image",
    "/media/pandora/Greger_group/Xinyao/PRRT Revision/PRRTm750 A1rb647 PSDck555 RIMgp488/261016 Repeat",
    "/media/pandora/Greger_group/Xinyao/PRRT Revision/PRRTm750 A1rb647 PSDck555 RIMgp488/261007 Repeat",
    "/media/pandora/Greger_group/Xinyao/PRRT Revision/PRRTm750 A1rb647 PSDck555 RIMgp488/261020 Repeat",
]

for data_path in folder:
    for filename in Path(data_path).glob("**/*.czi"):
        print(filename)
        try:
            synxm.process(filename, output_folder, parameters)
        except Exception as e:
            print(e)
            print("fail")

Reload all results

In [ ]:
import pandas as pd

sst = pd.concat(
    [
        pd.read_csv(x, index_col=0).assign(file=x.stem)
        for x in output_folder.glob("*_s_stats.csv")
    ],
    ignore_index=True,
)
points = pd.concat(
    [
        pd.read_csv(x, index_col=0).assign(file=x.stem)
        for x in output_folder.glob("*_points.csv")
    ],
    ignore_index=True,
)


In [ ]:
from synxm.plot import make_the_bagel_plot
import matplotlib.pyplot as plt
import yaml

gst = []
for x in output_folder.glob("*_g_stats.yaml"):
    with open(x, "r") as f:
        gst.append(yaml.safe_load(f))
        gst[-1]["file"] = x.stem
gst = pd.DataFrame.from_records(gst)
gst.to_csv(output_folder / "global_stats.csv")


make_the_bagel_plot(gst.sum().to_dict(), sst)
plt.savefig(output_folder / "bagel.pdf")

In [ ]:
from synxm.plot import make_the_histogram_distance

x = make_the_histogram_distance(points)

In [ ]:
df.to_csv("results/distance_prrt_psd.csv", index=False)

In [ ]:
import seaborn as sns

sns.catplot(
    gst.melt(id_vars="file"),
    y="file",
    x="value",
    col="variable",
    col_wrap=5,
    kind="bar",
    sharex=False,
).set_titles("{col_name}")